In [ ]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')          # Headless rendering (safe for servers/Colab)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import seaborn as sns

import sklearn

import warnings
import os
import pickle

warnings.filterwarnings('ignore')

In [26]:
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
DATA_PATH   = "Battery_dataset.csv"   # path to CSV
OUTPUT_DIR  = "outputs"                    # where plots & model get saved
RANDOM_SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [28]:
# loading the dataset
print("=" * 60)
print("BATTERY RUL PREDICTION MODEL")
print("=" * 60)

df = pd.read_csv(DATA_PATH)
df = df.sort_values(['battery_id', 'cycle']).reset_index(drop=True)

print(f"\n[1] Data loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"    Batteries : {df['battery_id'].unique().tolist()}")
for bid in df['battery_id'].unique():
    sub = df[df['battery_id'] == bid]
    print(f"    {bid} → {len(sub)} cycles  |  "
          f"Initial cap: {sub['BCt'].max():.4f} Ah  |  "
          f"Final SOH: {sub['SOH'].iloc[-1]:.2f}%")


BATTERY RUL PREDICTION MODEL

[1] Data loaded: 680 rows × 11 columns
    Batteries : ['B5', 'B6', 'B7']
    B5 → 220 cycles  |  Initial cap: 1.9862 Ah  |  Final SOH: 44.91%
    B6 → 210 cycles  |  Initial cap: 1.9937 Ah  |  Final SOH: 47.11%
    B7 → 250 cycles  |  Initial cap: 1.9909 Ah  |  Final SOH: 37.55%


In [29]:
df.head(5) # BCt battery capacity hai

,battery_id,cycle,chI,chV,chT,disI,disV,disT,BCt,SOH,RUL
0,B5,1,1.440147,4.254682,23.988733,1.894407,3.273523,32.980834,1.986196,99.309790,219
1,B5,2,1.416595,4.159825,25.665347,1.829949,4.038741,32.257920,1.986240,99.311985,218
2,B5,3,1.420272,4.276323,25.407910,1.942105,3.214433,35.134801,1.984252,99.212608,217
3,B5,4,1.337680,4.236697,27.069757,2.073577,3.134529,32.082988,1.969236,98.461812,216
4,B5,5,1.263946,4.142791,26.478353,2.049885,3.729341,32.483154,1.974862,98.743106,215


In [30]:
df.columns

Index(['battery_id', 'cycle', 'chI', 'chV', 'chT', 'disI', 'disV', 'disT',
       'BCt', 'SOH', 'RUL'],
      dtype='object')

In [31]:
print("\n[2] Engineering features...")

for bid in df['battery_id'].unique():
    mask = df['battery_id'] == bid

    # Rate of change features (first-order differences)
    df.loc[mask, 'capacity_fade_rate'] = df.loc[mask, 'BCt'].diff().fillna(0)
    df.loc[mask, 'SOH_rate']           = df.loc[mask, 'SOH'].diff().fillna(0)
    df.loc[mask, 'temp_rise_rate']     = df.loc[mask, 'chT'].diff().fillna(0)

    # Physics-grounded proxy feature
    df.loc[mask, 'voltage_drop']       = df.loc[mask, 'chV'] - df.loc[mask, 'disV']

    # Smoothed trend features (noise reduction)
    df.loc[mask, 'BCt_rolling_mean']   = df.loc[mask, 'BCt'].rolling(5, min_periods=1).mean()
    df.loc[mask, 'SOH_rolling_mean']   = df.loc[mask, 'SOH'].rolling(5, min_periods=1).mean()

ENGINEERED_FEATURES = [
    'capacity_fade_rate', 'SOH_rate', 'temp_rise_rate',
    'voltage_drop', 'BCt_rolling_mean', 'SOH_rolling_mean'
]
print(f"    Created {len(ENGINEERED_FEATURES)} engineered features: {ENGINEERED_FEATURES}")



[2] Engineering features...
    Created 6 engineered features: ['capacity_fade_rate', 'SOH_rate', 'temp_rise_rate', 'voltage_drop', 'BCt_rolling_mean', 'SOH_rolling_mean']


In [32]:
FEATURES = [
    # Raw sensor readings
    'chI',          # charge current
    'chV',          # charge voltage
    'chT',          # charge temperature
    'disI',         # discharge current
    'disV',         # discharge voltage
    'disT',         # discharge temperature
    'BCt',          # battery capacity (Ah)
    'SOH',          # state of health (%)
    # Engineered features
    'capacity_fade_rate',
    'SOH_rate',
    'temp_rise_rate',
    'voltage_drop',
    'BCt_rolling_mean',
    'SOH_rolling_mean',
]

TARGET = 'RUL'

print(f"\n[3] Feature set defined: {len(FEATURES)} features")
print(f"    Target: {TARGET}")
print("\n[4] Validation strategy: Leave-One-Battery-Out (LOBO)")



[3] Feature set defined: 14 features
    Target: RUL

[4] Validation strategy: Leave-One-Battery-Out (LOBO)


In [33]:
def build_models():
    return {
        'Linear Regression': LinearRegression(),
        'Ridge Regression':  Ridge(alpha=10.0),
        'Random Forest':     RandomForestRegressor(
                                 n_estimators=300,
                                 max_depth=12,
                                 random_state=RANDOM_SEED
                             ),
        'Gradient Boosting': GradientBoostingRegressor(
                                 n_estimators=300,
                                 max_depth=4,
                                 learning_rate=0.05,
                                 random_state=RANDOM_SEED
                             ),
    }

print("\n[5] Models defined:")
for name in build_models():
    print(f"    • {name}")


[5] Models defined:
    • Linear Regression
    • Ridge Regression
    • Random Forest
    • Gradient Boosting


In [34]:

print("\n[6] Training — Leave-One-Battery-Out Cross-Validation")
print("-" * 60)

BATTERIES    = ['B5', 'B6', 'B7']
all_results  = {}   # {battery: {model: {metric: value}}}
all_preds    = {}   # {battery: {model: predictions array}}
all_actual   = {}   # {battery: actual RUL array}
all_cycles   = {}   # {battery: cycle array}

for test_bat in BATTERIES:
    train_df = df[df['battery_id'] != test_bat]
    test_df  = df[df['battery_id'] == test_bat]

    X_train = train_df[FEATURES]
    y_train = train_df[TARGET]
    X_test  = test_df[FEATURES]
    y_test  = test_df[TARGET]

    all_actual[test_bat] = y_test.values
    all_cycles[test_bat] = test_df['cycle'].values

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    models = build_models()
    bat_results = {}
    bat_preds   = {}

    for name, model in models.items():
        # Choose scaled or raw features depending on model type
        if name in ['Linear Regression', 'Ridge Regression']:
            model.fit(X_train_s, y_train)
            preds = model.predict(X_test_s)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        preds = np.clip(preds, 0, None)   # RUL cannot be negative

        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2   = r2_score(y_test, preds)

        bat_results[name] = {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)}
        bat_preds[name]   = preds

    all_results[test_bat] = bat_results
    all_preds[test_bat]   = bat_preds

    print(f"\n  Test Battery: {test_bat}  (trained on {[b for b in BATTERIES if b != test_bat]})")
    print(f"  {'Model':<24} {'MAE':>7} {'RMSE':>7} {'R²':>8}")
    print(f"  {'-'*50}")
    for name, res in bat_results.items():
        print(f"  {name:<24} {res['MAE']:>7.2f} {res['RMSE']:>7.2f} {res['R²']:>8.4f}")



[6] Training — Leave-One-Battery-Out Cross-Validation
------------------------------------------------------------

  Test Battery: B5  (trained on ['B6', 'B7'])
  Model                        MAE    RMSE       R²
  --------------------------------------------------
  Linear Regression          11.45   11.97   0.9645
  Ridge Regression           11.42   12.01   0.9643
  Random Forest              10.65   12.20   0.9631
  Gradient Boosting          11.70   14.09   0.9508

  Test Battery: B6  (trained on ['B5', 'B7'])
  Model                        MAE    RMSE       R²
  --------------------------------------------------
  Linear Regression          25.72   25.79   0.8190
  Ridge Regression           25.64   25.73   0.8198
  Random Forest              25.09   25.48   0.8233
  Gradient Boosting          25.12   25.89   0.8176

  Test Battery: B7  (trained on ['B5', 'B6'])
  Model                        MAE    RMSE       R²
  --------------------------------------------------
  Linear Reg

In [35]:

print("\n\n[7] Model Summary — Averaged Across All Batteries")
print("-" * 60)

MODEL_NAMES = list(build_models().keys())
avg_summary = {}

for name in MODEL_NAMES:
    maes  = [all_results[b][name]['MAE']  for b in BATTERIES]
    rmses = [all_results[b][name]['RMSE'] for b in BATTERIES]
    r2s   = [all_results[b][name]['R²']   for b in BATTERIES]
    avg_summary[name] = {
        'Avg MAE':  round(np.mean(maes),  2),
        'Avg RMSE': round(np.mean(rmses), 2),
        'Avg R²':   round(np.mean(r2s),   4),
    }
    print(f"  {name:<24}  Avg MAE={np.mean(maes):.2f}  "
          f"Avg RMSE={np.mean(rmses):.2f}  Avg R²={np.mean(r2s):.4f}")

best_model_name = min(avg_summary, key=lambda k: avg_summary[k]['Avg RMSE'])
print(f"\n  ★  Best Model: {best_model_name}  "
      f"(Avg RMSE={avg_summary[best_model_name]['Avg RMSE']:.2f}, "
      f"Avg R²={avg_summary[best_model_name]['Avg R²']:.4f})")




[7] Model Summary — Averaged Across All Batteries
------------------------------------------------------------
  Linear Regression         Avg MAE=23.20  Avg RMSE=23.67  Avg R²=0.8571
  Ridge Regression          Avg MAE=23.16  Avg RMSE=23.66  Avg R²=0.8573
  Random Forest             Avg MAE=22.59  Avg RMSE=23.59  Avg R²=0.8587
  Gradient Boosting         Avg MAE=22.98  Avg RMSE=24.38  Avg R²=0.8524

  ★  Best Model: Random Forest  (Avg RMSE=23.59, Avg R²=0.8587)


In [36]:
print("\n\n[8] Battery Health Comparison")
print("-" * 60)

battery_health = {}
for bid in BATTERIES:
    sub = df[df['battery_id'] == bid]
    fade_pct = (sub['BCt'].max() - sub['BCt'].min()) / sub['BCt'].max() * 100
    battery_health[bid] = {
        'Total Cycles':     int(sub['cycle'].max()),
        'Initial Cap (Ah)': round(sub['BCt'].max(), 4),
        'Final Cap (Ah)':   round(sub['BCt'].min(), 4),
        'Cap Fade (%)':     round(fade_pct, 2),
        'Avg SOH (%)':      round(sub['SOH'].mean(), 2),
        'Final SOH (%)':    round(sub['SOH'].iloc[-1], 2),
    }
    print(f"\n  {bid}:")
    for k, v in battery_health[bid].items():
        print(f"    {k:<22}: {v}")

# Ranking logic
fades  = {b: battery_health[b]['Cap Fade (%)'] for b in BATTERIES}
avgsoh = {b: battery_health[b]['Avg SOH (%)']  for b in BATTERIES}
best_bat = min(fades, key=fades.get)   # least fade = best

print(f"\n  ★  Best Battery: {best_bat}")
print(f"     Reason: Lowest capacity fade ({fades[best_bat]:.1f}%) "
      f"and highest average SOH ({avgsoh[best_bat]:.1f}%)")




[8] Battery Health Comparison
------------------------------------------------------------

  B5:
    Total Cycles          : 220
    Initial Cap (Ah)      : 1.9862
    Final Cap (Ah)        : 0.8983
    Cap Fade (%)          : 54.78
    Avg SOH (%)           : 72.39
    Final SOH (%)         : 44.91

  B6:
    Total Cycles          : 210
    Initial Cap (Ah)      : 1.9937
    Final Cap (Ah)        : 0.9422
    Cap Fade (%)          : 52.74
    Avg SOH (%)           : 73.65
    Final SOH (%)         : 47.11

  B7:
    Total Cycles          : 250
    Initial Cap (Ah)      : 1.9909
    Final Cap (Ah)        : 0.751
    Cap Fade (%)          : 62.28
    Avg SOH (%)           : 68.63
    Final SOH (%)         : 37.55

  ★  Best Battery: B6
     Reason: Lowest capacity fade (52.7%) and highest average SOH (73.7%)


In [37]:

print("\n\n[9] Feature Importance (Random Forest trained on B5+B6)")
print("-" * 60)

train_all_df = df[df['battery_id'].isin(['B5', 'B6'])]
rf_importance = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_SEED)
rf_importance.fit(train_all_df[FEATURES], train_all_df[TARGET])

importances = pd.Series(rf_importance.feature_importances_, index=FEATURES).sort_values(ascending=False)
for feat, imp in importances.items():
    bar = '█' * int(imp * 100)
    tag = ' ← ENGINEERED' if feat in ENGINEERED_FEATURES else ''
    print(f"  {feat:<24} {imp:.4f}  {bar}{tag}")




[9] Feature Importance (Random Forest trained on B5+B6)
------------------------------------------------------------
  SOH                      0.2800  ███████████████████████████
  BCt_rolling_mean         0.2635  ██████████████████████████ ← ENGINEERED
  BCt                      0.2372  ███████████████████████
  SOH_rolling_mean         0.2148  █████████████████████ ← ENGINEERED
  chI                      0.0007  
  disI                     0.0005  
  disT                     0.0005  
  chV                      0.0005  
  chT                      0.0005  
  disV                     0.0005  
  voltage_drop             0.0004   ← ENGINEERED
  temp_rise_rate           0.0004   ← ENGINEERED
  capacity_fade_rate       0.0002   ← ENGINEERED
  SOH_rate                 0.0002   ← ENGINEERED


In [38]:

print("\n\n[10] Saving final model...")

# Train final model on ALL available data
X_all = df[FEATURES]
y_all = df[TARGET]

final_rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_SEED)
final_rf.fit(X_all, y_all)

final_scaler = StandardScaler()
final_scaler.fit(X_all)   # fit scaler on full data for reference

model_package = {
    'model':    final_rf,
    'scaler':   final_scaler,
    'features': FEATURES,
    'metadata': {
        'model_type':    'RandomForestRegressor',
        'n_estimators':  300,
        'max_depth':     12,
        'avg_r2_lobo':   avg_summary['Random Forest']['Avg R²'],
        'avg_mae_lobo':  avg_summary['Random Forest']['Avg MAE'],
        'avg_rmse_lobo': avg_summary['Random Forest']['Avg RMSE'],
        'best_battery':  best_bat,
        'trained_on':    'B5, B6, B7 (all data)',
    }
}

model_path = os.path.join(OUTPUT_DIR, 'battery_rul_rf_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_package, f)

print(f"    Model saved to: {model_path}")
print(f"    To load: model_pkg = pickle.load(open('{model_path}', 'rb'))")




[10] Saving final model...
    Model saved to: outputs\battery_rul_rf_model.pkl
    To load: model_pkg = pickle.load(open('outputs\battery_rul_rf_model.pkl', 'rb'))


In [39]:

print("\n\n[12] Generating visualisation plots...")

COLORS = {'B5': '#2196F3', 'B6': '#4CAF50', 'B7': '#FF5722'}
LIGHT  = {'B5': '#BBDEFB', 'B6': '#C8E6C9', 'B7': '#FFCCBC'}
sns.set_theme(style='whitegrid')

fig = plt.figure(figsize=(22, 26), facecolor='white')
fig.suptitle(
    'Battery RUL Prediction — Hybrid Physics-Informed ML\n'
    'Internship Project  |  Avijnan Purkait',
    fontsize=20, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.48, wspace=0.35,
                       top=0.94, bottom=0.03, left=0.06, right=0.97)



[12] Generating visualisation plots...


In [40]:
# ── Plot 1: Capacity Fade ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
for bid in BATTERIES:
    sub = df[df['battery_id'] == bid]
    ax1.plot(sub['cycle'], sub['BCt'], color=COLORS[bid], lw=2.5, label=bid)
    ax1.fill_between(sub['cycle'], sub['BCt'], alpha=0.07, color=COLORS[bid])
ax1.axhline(1.4, color='red', ls='--', lw=1.8, alpha=0.75, label='~70% capacity (typical EOL)')
ax1.set_title('Battery Capacity Fade over Charge-Discharge Cycles', fontsize=14, fontweight='bold')
ax1.set_xlabel('Cycle Number'); ax1.set_ylabel('Capacity (Ah)')
ax1.legend(fontsize=10); ax1.set_facecolor('#F5F5F5')
ax1.annotate('B7: fastest degradation\n(62.3% total fade)',
             xy=(220, 0.95), fontsize=9, color=COLORS['B7'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLORS['B7']), xytext=(165, 1.3))
ax1.annotate('B6: slowest degradation\n(52.7% total fade — BEST)',
             xy=(160, 1.48), fontsize=9, color=COLORS['B6'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLORS['B6']), xytext=(60, 1.7))


Text(60, 1.7, 'B6: slowest degradation\n(52.7% total fade — BEST)')

In [41]:
# ── Plot 2: SOH Degradation ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :2])
for bid in BATTERIES:
    sub = df[df['battery_id'] == bid]
    ax2.plot(sub['cycle'], sub['SOH'], color=COLORS[bid], lw=2, label=bid)
ax2.axhline(80, color='orange', ls='--', lw=1.5, label='80% SOH (typical EOL threshold)')
ax2.set_title('State of Health (SOH) Degradation', fontsize=13, fontweight='bold')
ax2.set_xlabel('Cycle'); ax2.set_ylabel('SOH (%)'); ax2.legend(); ax2.set_facecolor('#F5F5F5')


In [42]:
# ── Plot 3: Feature Importance ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
top_fi = importances.head(8)
bar_colors = ['#1565C0' if f in ENGINEERED_FEATURES else '#90CAF9' for f in top_fi.index[::-1]]
bars = ax3.barh(top_fi.index[::-1], top_fi.values[::-1], color=bar_colors)
ax3.set_title('Feature Importance\n(Random Forest)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Importance')
for bar, val in zip(bars, top_fi.values[::-1]):
    ax3.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=8)
# Legend
from matplotlib.patches import Patch
ax3.legend(handles=[Patch(color='#1565C0', label='Engineered'),
                    Patch(color='#90CAF9', label='Raw sensor')], fontsize=8)
ax3.set_facecolor('#F5F5F5')

In [43]:
# ── Plots 4-6: RUL Actual vs Predicted per battery ───────────────────────────
for i, bid in enumerate(BATTERIES):
    ax = fig.add_subplot(gs[2, i])
    cycles = all_cycles[bid]
    actual = all_actual[bid]
    rf_pred = all_preds[bid]['Random Forest']
    ax.plot(cycles, actual,  color=COLORS[bid], lw=2.5, label='Actual RUL')
    ax.plot(cycles, rf_pred, color='black', lw=1.8, ls='--', alpha=0.85, label='Predicted RUL')
    ax.fill_between(cycles, actual, rf_pred, alpha=0.12, color='gray', label='Error')
    res = all_results[bid]['Random Forest']
    ax.set_title(f'{bid} — RUL Prediction\nMAE={res["MAE"]:.1f}  RMSE={res["RMSE"]:.1f}  R²={res["R²"]:.3f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Cycle'); ax.set_ylabel('RUL (cycles)')
    ax.legend(fontsize=8); ax.set_facecolor('#F5F5F5')


In [44]:
# ── Plot 7: Model Comparison ──────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[3, :2])
x   = np.arange(len(MODEL_NAMES))
w   = 0.28
avg_maes  = [avg_summary[n]['Avg MAE']  for n in MODEL_NAMES]
avg_rmses = [avg_summary[n]['Avg RMSE'] for n in MODEL_NAMES]
avg_r2s   = [avg_summary[n]['Avg R²']   for n in MODEL_NAMES]
short_names = ['Linear\nRegression', 'Ridge\nRegression', 'Random\nForest', 'Gradient\nBoosting']
b1 = ax7.bar(x - w/2, avg_maes,  w, label='Avg MAE',  color='#1565C0', alpha=0.85)
b2 = ax7.bar(x + w/2, avg_rmses, w, label='Avg RMSE', color='#E65100', alpha=0.85)
for b, v in zip(b1, avg_maes):  ax7.text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=8)
for b, v in zip(b2, avg_rmses): ax7.text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=8)
ax7.set_xticks(x); ax7.set_xticklabels(short_names, fontsize=10)
ax7.set_title('Model Comparison — Avg MAE & RMSE (LOBO Validation)', fontsize=12, fontweight='bold')
ax7.set_ylabel('Error (cycles)'); ax7.legend(); ax7.set_facecolor('#F5F5F5')
ax7b = ax7.twinx()
ax7b.plot(x, avg_r2s, 'D--', color='#2E7D32', ms=9, lw=2, label='Avg R²')
for xi, rv in zip(x, avg_r2s):
    ax7b.annotate(f'R²={rv:.3f}', (xi, rv), xytext=(0, 10),
                  textcoords='offset points', ha='center', fontsize=8,
                  color='#2E7D32', fontweight='bold')
ax7b.set_ylabel('R²', color='#2E7D32'); ax7b.set_ylim(0.7, 1.05); ax7b.legend(loc='upper right')


In [45]:
# ── Plot 8: Battery Scorecard ─────────────────────────────────────────────────
ax8 = fig.add_subplot(gs[3, 2])
ax8.axis('off')
ax8.set_title('Battery Scorecard', fontsize=13, fontweight='bold', pad=10)
y_pos = 0.88
from matplotlib.patches import FancyBboxPatch
for bid in BATTERIES:
    s = battery_health[bid]
    rect = FancyBboxPatch((0.02, y_pos-0.27), 0.96, 0.28,
                          boxstyle="round,pad=0.02", lw=2,
                          edgecolor=COLORS[bid], facecolor=LIGHT[bid], alpha=0.55,
                          transform=ax8.transAxes)
    ax8.add_patch(rect)
    ax8.text(0.10, y_pos-0.06, bid, transform=ax8.transAxes,
             fontsize=16, fontweight='bold', color=COLORS[bid])
    ax8.text(0.33, y_pos-0.04, f"Cycles: {s['Total Cycles']}   Fade: {s['Cap Fade (%)']}%",
             transform=ax8.transAxes, fontsize=9, color='#212121')
    ax8.text(0.33, y_pos-0.15, f"Avg SOH: {s['Avg SOH (%)']}%   Final SOH: {s['Final SOH (%)']}%",
             transform=ax8.transAxes, fontsize=9, color='#757575')
    if bid == best_bat:
        ax8.text(0.78, y_pos-0.09, '⭐ BEST', transform=ax8.transAxes,
                 fontsize=11, fontweight='bold', color='#2E7D32')
    y_pos -= 0.32

In [46]:
# ── Plot 9: Discharge Temperature Trend ──────────────────────────────────────
ax9 = fig.add_subplot(gs[4, :2])
for bid in BATTERIES:
    sub  = df[df['battery_id'] == bid]
    roll = sub['disT'].rolling(10, min_periods=1).mean()
    ax9.plot(sub['cycle'], roll, color=COLORS[bid], lw=2, label=bid)
ax9.set_title('Discharge Temperature Trend (10-cycle rolling mean) — Thermal Aging Proxy',
              fontsize=12, fontweight='bold')
ax9.set_xlabel('Cycle'); ax9.set_ylabel('Temperature (°C)')
ax9.legend(); ax9.set_facecolor('#F5F5F5')


In [49]:
# ── Plot 10: Voltage Drop (Internal Resistance Proxy) ─────────────────────────
ax10 = fig.add_subplot(gs[4, 2])
for bid in BATTERIES:
    sub = df[df['battery_id'] == bid]
    ax10.plot(sub['cycle'], sub['voltage_drop'], color=COLORS[bid], lw=1.5, alpha=0.75, label=bid)
ax10.set_title('Voltage Drop (chV − disV)\nInternal Resistance Proxy', fontsize=12, fontweight='bold')
ax10.set_xlabel('Cycle'); ax10.set_ylabel('Voltage Drop (V)')
ax10.legend(); ax10.set_facecolor('#F5F5F5')

plot_path = os.path.join(OUTPUT_DIR, 'battery_rul_analysis.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()          # Display the graph

plt.close()
print(f"    Plot saved to: {plot_path}")

    Plot saved to: outputs\battery_rul_analysis.png


In [50]:
# final summary
print("\n")
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\n  BEST MODEL   : {best_model_name}")
print(f"  Avg MAE      : {avg_summary[best_model_name]['Avg MAE']} cycles")
print(f"  Avg RMSE     : {avg_summary[best_model_name]['Avg RMSE']} cycles")
print(f"  Avg R²       : {avg_summary[best_model_name]['Avg R²']}")
print(f"\n  BEST BATTERY : {best_bat}")
print(f"  Reason       : Lowest capacity fade ({fades[best_bat]:.1f}%) "
      f"and highest average SOH ({avgsoh[best_bat]:.1f}%)")
print(f"\n  Outputs saved in: {OUTPUT_DIR}/")
print(f"    → battery_rul_rf_model.pkl  (trained model)")
print(f"    → battery_rul_analysis.png  (full dashboard plot)")
print("\n" + "=" * 60)




FINAL SUMMARY

  BEST MODEL   : Random Forest
  Avg MAE      : 22.59 cycles
  Avg RMSE     : 23.59 cycles
  Avg R²       : 0.8587

  BEST BATTERY : B6
  Reason       : Lowest capacity fade (52.7%) and highest average SOH (73.7%)

  Outputs saved in: outputs/
    → battery_rul_rf_model.pkl  (trained model)
    → battery_rul_analysis.png  (full dashboard plot)

